In [11]:
# ==========================================
# Cell 1: 基础配置 (V1 - Spatial Transfer Learning)
# ==========================================
import os, pathlib, json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = pathlib.Path('.')
DATA_DIR = ROOT / 'data' 

# 专门为空间迁移实验新建输出目录，避免覆盖之前的全局实验结果
OUT_DIR = ROOT / 'output_spatial_1'
CM_DIR = OUT_DIR / 'confusion_matrices'
MODEL_DIR = OUT_DIR / 'models'

for d in [CM_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print(f'Device: {DEVICE}')
print(f'Output Base Dir: {OUT_DIR}')

Device: cuda
Output Base Dir: output_spatial_1


In [12]:
# ==========================================
# Cell 2: 数据读取与 Kalman Filter 核心逻辑
# ==========================================
stations_list_path = DATA_DIR / 'stations_list'
station_ids = [int(x) for x in stations_list_path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]').split() if x.isdigit()]
TOTAL_STATIONS = len(station_ids)

def parse_day_matrix(line: str):
    line = line.strip()
    if line.startswith('[') and line.endswith(']'): line = line[1:-1]
    rows = [r.strip() for r in line.split(';') if r.strip()]
    data = []
    for r in rows:
        nums = [float(x) for x in r.split() if x.replace('.', '', 1).isdigit()]
        if nums: data.append(nums)
    if not data: return None
    arr = np.full((len(data), max(len(rr) for rr in data)), np.nan, dtype=float)
    for i, rr in enumerate(data): arr[i, :len(rr)] = rr
    return arr

def iter_day_matrices(path: pathlib.Path):
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            mat = parse_day_matrix(line)
            yield mat if mat is not None else None

def load_labels(path: pathlib.Path):
    txt = path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    return np.array([int(x) for x in txt.split() if x.isdigit()], dtype=int)

labels_train = load_labels(DATA_DIR / 'PEMS_trainlabels')
labels_test  = load_labels(DATA_DIR / 'PEMS_testlabels')

# --- 核心：一维卡尔曼滤波函数 (1D Kalman Filter) ---
def apply_kalman_filter(curve, process_variance=1e-4, measurement_variance=1e-2):
    """使用卡尔曼滤波对交通流一维时序数据进行软去噪"""
    n = len(curve)
    xhat = np.zeros(n)      
    P = np.zeros(n)         
    xhat[0] = curve[0]
    P[0] = 1.0
    
    for k in range(1, n):
        xhat_minus = xhat[k-1]
        P_minus = P[k-1] + process_variance
        K = P_minus / (P_minus + measurement_variance) 
        xhat[k] = xhat_minus + K * (curve[k] - xhat_minus)
        P[k] = (1 - K) * P_minus
        
    return xhat

In [13]:
# ==========================================
# Cell 3: 构建支持空间隔离的 Dataset (带 ID 打印版)
# ==========================================
def _process_curve_with_kalman(curve: np.ndarray):
    curve = curve[:144]
    if curve.shape[0] < 144:
        curve = np.pad(curve, (0, 144 - curve.shape[0]), mode='constant', constant_values=np.nan)
    
    idx = np.arange(curve.size)
    mask = ~np.isnan(curve)
    if mask.any(): 
        curve = np.interp(idx, idx[mask], curve[mask])
    curve = np.nan_to_num(curve, nan=0.0)
    
    curve = apply_kalman_filter(curve)

    min_val, max_val = curve.min(), curve.max()
    denom = max_val - min_val + 1e-8
    norm_curve = (curve - min_val) / denom
    
    mean_val, std_val = np.mean(norm_curve), np.std(norm_curve)
    log_max_val = np.log1p(max_val) / 10.0 
    stats = np.array([mean_val, std_val, log_max_val], dtype=np.float32)

    grad_curve = np.gradient(norm_curve)
    return np.stack([norm_curve, grad_curve], axis=0).astype(np.float32), stats

class PEMS_SpatialDataset(Dataset):
    def __init__(self, split_path, labels, target_indices, augment=False, split_name=""):
        self.samples, self.augment, self.split_name = [], augment, split_name
        self.target_indices = set(target_indices)
        
        # 【新增】：用于收集当前实例实际加载的物理 Station ID
        self.loaded_station_ids = set()

        for day_i, mat in enumerate(iter_day_matrices(split_path)):
            if day_i >= len(labels): break
            
            if mat is None or mat.shape[0] != TOTAL_STATIONS or mat.shape[1] < 144: 
                continue
                
            y = int(labels[day_i]) - 1
            if y < 0 or y > 6: continue

            for local_sidx in range(TOTAL_STATIONS):
                if local_sidx not in self.target_indices:
                    continue

                # 提取并记录真实的 station_id
                actual_station_id = int(station_ids[local_sidx])
                self.loaded_station_ids.add(actual_station_id)

                two_channel_seq, stats = _process_curve_with_kalman(mat[local_sidx, :144])
                if not np.isfinite(two_channel_seq).all(): continue
                meta = {
                    "station_id": actual_station_id,
                    "station_pos": int(local_sidx),
                    "day_i": int(day_i), "split": self.split_name,
                }
                self.samples.append((two_channel_seq, stats, y, meta))
                
        self.n = len(self.samples)
        print(f"[{split_name}] 空间切片加载完成，包含 {len(self.target_indices)} 个目标站点，共提取 {self.n} 条样本。")
        
        # 【新增】：在加载结束后，排序并打印出所有的物理 ID
        sorted_ids = sorted(list(self.loaded_station_ids))
        print(f"[{split_name}] 涉及的原始物理 Station IDs 列表 (共 {len(sorted_ids)} 个):")
        
        # 为了不占太多行，这里按照每行打印 15 个 ID 进行排版
        for i in range(0, len(sorted_ids), 15):
            print("  " + " ".join([str(id) for id in sorted_ids[i:i+15]]))
        print("-" * 50) # 打印分割线

    def __len__(self): return self.n
    def __getitem__(self, idx):
        seq, stats, y, meta = self.samples[idx]
        seq_tensor = torch.from_numpy(seq)
        if self.augment:
            seq_tensor = seq_tensor * random.uniform(0.95, 1.05)
            shifts = random.randint(-2, 2)
            if shifts != 0: seq_tensor = torch.roll(seq_tensor, shifts=shifts, dims=1)
            seq_tensor += torch.randn_like(seq_tensor) * 0.005
        return seq_tensor, torch.from_numpy(stats), torch.tensor(y, dtype=torch.long), meta

In [14]:
# ==========================================
# Cell 4: 1D-CNN + BiLSTM 模型架构
# ==========================================
class CNN_BiLSTM_Model(nn.Module):
    def __init__(self, in_channels=2, num_classes=7):
        super().__init__()
        # 1. CNN 提取局部突变特征
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout1d(0.1),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout1d(0.1)
        )
        
        # 2. BiLSTM 提取时序长程记忆
        self.lstm = nn.LSTM(
            input_size=64, hidden_size=64, 
            num_layers=2, batch_first=True, bidirectional=True, dropout=0.2
        )
        
        # 3. 注意力/池化与分类
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Dropout(0.3), 
            nn.Linear(128 + 3, 64), 
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x, stats):
        cnn_out = self.cnn(x) 
        lstm_in = cnn_out.permute(0, 2, 1) 
        lstm_out, _ = self.lstm(lstm_in) 
        
        lstm_out_pool = lstm_out.permute(0, 2, 1) 
        features = self.gap(lstm_out_pool).view(lstm_out_pool.size(0), -1) 
        
        combined = torch.cat([features, stats], dim=1) 
        return self.fc(combined)

In [15]:
# ==========================================
# Cell 5: 划分训练空间(0-199)与未知测试空间(200-249)
# ==========================================
train_path = DATA_DIR / 'PEMS_train'
test_path  = DATA_DIR / 'PEMS_test'

# 定义空间站点的物理索引范围
TRAIN_STATIONS = list(range(0, 200))    # 0-199 站点用于模型训练
TEST_STATIONS = list(range(200, 250))   # 200-249 站点作为全新空间节点进行零样本测试

print("\n>>> 开始加载空间隔离的交通流数据集...")

# 训练集和验证集仅限于 0-199 站点
ds_train_spatial_source = PEMS_SpatialDataset(train_path, labels_train, target_indices=TRAIN_STATIONS, augment=True, split_name="train_spatial_200")
ds_val_spatial_source   = PEMS_SpatialDataset(train_path, labels_train, target_indices=TRAIN_STATIONS, augment=False, split_name="val_spatial_200")

# 终极测试集：只从测试文件中读取 200-249 站点的数据
ds_test_spatial_unseen  = PEMS_SpatialDataset(test_path, labels_test, target_indices=TEST_STATIONS, augment=False, split_name="test_spatial_unseen_50")

# 对训练数据进行 80/20 随机划分
spatial_total_len = len(ds_train_spatial_source)
indices = torch.randperm(spatial_total_len, generator=torch.Generator().manual_seed(42)).tolist()
train_subset_spatial = Subset(ds_train_spatial_source, indices[:int(spatial_total_len * 0.8)])
val_subset_spatial   = Subset(ds_val_spatial_source, indices[int(spatial_total_len * 0.8):])

spatial_loaders = {
    'train': DataLoader(train_subset_spatial, batch_size=256, shuffle=True, num_workers=0),
    'val':   DataLoader(val_subset_spatial, batch_size=512, shuffle=False, num_workers=0),
    'test_unseen': DataLoader(ds_test_spatial_unseen, batch_size=512, shuffle=False, num_workers=0),
}


>>> 开始加载空间隔离的交通流数据集...


[train_spatial_200] 空间切片加载完成，包含 200 个目标站点，共提取 25000 条样本。
[train_spatial_200] 涉及的原始物理 Station IDs 列表 (共 200 个):
  400000 400001 400009 400010 400015 400017 400025 400026 400027 400030 400031 400035 400037 400039 400040
  400041 400043 400044 400045 400049 400052 400053 400057 400059 400060 400065 400067 400071 400073 400074
  400075 400078 400079 400082 400083 400085 400086 400088 400090 400091 400093 400094 400095 400096 400097
  400100 400103 400107 400108 400109 400110 400113 400115 400116 400118 400122 400124 400125 400126 400127
  400132 400137 400138 400141 400143 400145 400147 400148 400149 400150 400152 400153 400154 400156 400158
  400160 400162 400164 400165 400169 400171 400172 400174 400180 400181 400182 400183 400184 400185 400186
  400189 400190 400192 400193 400195 400201 400202 400203 400204 400206 400208 400209 400211 400212 400213
  400214 400216 400218 400221 400222 400223 400224 400225 400227 400228 400231 400232 400235 400236 400237
  400238 400240 400242 400248 400

In [16]:
# ==========================================
# Cell 6: 在 0-199 空间域上从头训练模型
# ==========================================
def train_spatial_model(loaders, epochs=60):
    model = CNN_BiLSTM_Model().to(DEVICE)
    # 初始化全新权重，坚守零样本迁移的实验道德
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-3) 
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    weights = torch.tensor([1.0, 2.0, 4.0, 2.0, 1.5, 1.0, 1.0]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
    
    best_acc, best_state, counter, patience = 0.0, None, 0, 15
    
    print("\n>>> 开始在 0-199 监测站点上训练模型基础感知能力...")
    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0
        for x, stats, y, _ in loaders['train']:
            x, stats, y = x.to(DEVICE), stats.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x, stats)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        scheduler.step() 
        
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, stats, y, _ in loaders['val']:
                x, stats, y = x.to(DEVICE), stats.to(DEVICE), y.to(DEVICE)
                preds = torch.argmax(model(x, stats), dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        
        val_acc = correct / total if total > 0 else 0
        if val_acc > best_acc:
            best_acc, best_state, counter = val_acc, model.state_dict(), 0
        else:
            counter += 1
            
        if epoch % 5 == 0:
            print(f"[Spatial Train] Ep {epoch}/{epochs} | Loss: {total_loss/len(loaders['train']):.4f} | Source Domain Val Acc: {val_acc:.2%} | LR: {scheduler.get_last_lr()[0]:.6f}")
        
        if counter >= patience:
            print(f"[Spatial Train] 早停触发于 epoch {epoch}")
            break
            
    return best_state, best_acc

spatial_model_state, spatial_val_acc = train_spatial_model(spatial_loaders, epochs=60)
print(f"\n源域(0-199站点)模型训练完成！验证准确率: {spatial_val_acc:.2%}")
if spatial_model_state:
    torch.save(spatial_model_state, MODEL_DIR / 'cnn_lstm_v400_spatial_source.pth')


>>> 开始在 0-199 监测站点上训练模型基础感知能力...


[Spatial Train] Ep 5/60 | Loss: 1.4752 | Source Domain Val Acc: 43.06% | LR: 0.000983
[Spatial Train] Ep 10/60 | Loss: 1.4303 | Source Domain Val Acc: 41.04% | LR: 0.000933
[Spatial Train] Ep 15/60 | Loss: 1.3894 | Source Domain Val Acc: 45.10% | LR: 0.000854
[Spatial Train] Ep 20/60 | Loss: 1.3700 | Source Domain Val Acc: 51.46% | LR: 0.000750
[Spatial Train] Ep 25/60 | Loss: 1.3517 | Source Domain Val Acc: 52.38% | LR: 0.000630
[Spatial Train] Ep 30/60 | Loss: 1.3309 | Source Domain Val Acc: 53.34% | LR: 0.000501
[Spatial Train] Ep 35/60 | Loss: 1.3145 | Source Domain Val Acc: 52.48% | LR: 0.000371
[Spatial Train] Ep 40/60 | Loss: 1.3072 | Source Domain Val Acc: 55.36% | LR: 0.000251
[Spatial Train] Ep 45/60 | Loss: 1.2958 | Source Domain Val Acc: 56.72% | LR: 0.000147
[Spatial Train] Ep 50/60 | Loss: 1.2898 | Source Domain Val Acc: 55.32% | LR: 0.000068
[Spatial Train] Ep 55/60 | Loss: 1.2876 | Source Domain Val Acc: 56.52% | LR: 0.000018
[Spatial Train] Ep 60/60 | Loss: 1.2823 | So

In [17]:
# ==========================================
# Cell 7: 泛化能力评估与混淆矩阵导出 (修复 Tensor 报错版)
# ==========================================
def evaluate_spatial_generalization(model_state, loader, dataset_name):
    model = CNN_BiLSTM_Model().to(DEVICE)
    model.load_state_dict(model_state)
    model.eval()

    y_true, y_pred, metas = [], [], []
    with torch.no_grad():
        for x, stats, y, meta in loader:
            x, stats = x.to(DEVICE), stats.to(DEVICE)
            logits = model(x, stats)
            preds = torch.argmax(logits, dim=1)
            
            y_true.extend(y.cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())

            if isinstance(meta, dict):
                bs = len(y)
                for i in range(bs):
                    # 【核心修复】：将 PyTorch Tensor 转回原生 Python 数据类型
                    meta_dict = {}
                    for k in meta:
                        val = meta[k][i]
                        meta_dict[k] = val.item() if isinstance(val, torch.Tensor) else val
                    metas.append(meta_dict)

    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-8)
    
    plt.figure(figsize=(10, 8))
    labels_day = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Oranges' if 'unseen' in dataset_name else 'Blues', xticklabels=labels_day, yticklabels=labels_day)
    plt.title(f"Zero-shot Spatial Transfer ({dataset_name})", fontsize=16, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    
    save_path = CM_DIR / f'spatial_transfer_{dataset_name}_v1.png'
    plt.savefig(save_path, dpi=300)
    print(f"\n[{dataset_name}] 混淆矩阵已保存: {save_path}")
    plt.close()
    
    rows = []
    for i in range(len(y_true)):
        m = metas[i]
        yt, yp = y_true[i], y_pred[i]
        rows.append({
            # 在这里多加一层 int() 保底，确保万无一失
            "station_pos": int(m.get("station_pos", -1)),
            "day_i": int(m.get("day_i", -1)), 
            "y_true": yt, "y_pred": yp,
            "true_label": labels_day[yt], "pred_label": labels_day[yp],
            "correct": bool(yt == yp),
        })

    df = pd.DataFrame(rows).sort_values(["correct","station_pos","day_i"]).reset_index(drop=True)
    out_csv = CM_DIR / f"predictions_{dataset_name}_v1.csv"
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    
    acc = df['correct'].mean()
    print(f"[{dataset_name}] 预测准确率: {acc:.2%}")
    return acc

print("\n>>> 开始进行模型鲁棒性终极评估...")
evaluate_spatial_generalization(spatial_model_state, spatial_loaders['val'], 'Source_Domain_Val_0_199')

# 真正的考验：测试模型在从未见过的 201-250 站点上的表现
unseen_acc = evaluate_spatial_generalization(spatial_model_state, spatial_loaders['test_unseen'], 'Target_Domain_Unseen_200_249')
print(f"\n==========================================")
print(f"实验结论：在面对全新监测站点时，模型的预测准确率为 {unseen_acc:.2%}。")
print(f"==========================================")


>>> 开始进行模型鲁棒性终极评估...



[Source_Domain_Val_0_199] 混淆矩阵已保存: output_spatial_1/confusion_matrices/spatial_transfer_Source_Domain_Val_0_199_v1.png
[Source_Domain_Val_0_199] 预测准确率: 57.08%

[Target_Domain_Unseen_200_249] 混淆矩阵已保存: output_spatial_1/confusion_matrices/spatial_transfer_Target_Domain_Unseen_200_249_v1.png
[Target_Domain_Unseen_200_249] 预测准确率: 62.91%

实验结论：在面对全新监测站点时，模型的预测准确率为 62.91%。
